In [1]:
!pip install langchain langgraph

In [2]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [3]:
# Define State
class CrashState(TypedDict):
  input: str
  step1: str
  step2: str
  step3: str

In [7]:
# Define Steps
def step_1(state: CrashState) -> CrashState:
  print("Step 1 Executed")
  return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
  print("Step 2 hanging... now manually interrupt from the notebook toolbar (STOP BUTTON)")
  time.sleep(30)
  return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
  print("Step 3 Executed")
  return {"step3": "done"}

In [12]:
builder = StateGraph(CrashState)

builder.add_node('step_1', step_1)
builder.add_node('step_2', step_2)
builder.add_node('step_3', step_3)


builder.add_edge(START, 'step_1')
builder.add_edge('step_1', 'step_2')
builder.add_edge('step_2', 'step_3')
builder.add_edge('step_3', END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [14]:
try:
  print("Running  graph: Please manually interrupt during step 2...")
  graph.invoke({"input": "start"}, config={"configurable": {"thread_id": "thread-1"}})
except KeyboardInterrupt:
  print("Kernel Interrupted")

Running  graph: Please manually interrupt during step 2...
Step 1 Executed
Step 2 hanging... now manually interrupt from the notebook toolbar (STOP BUTTON)
Kernel Interrupted


In [15]:
graph.get_state({"configurable": {"thread_id": 'thread-1'}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18598b-49e0-608c-8006-5e9734cc65e0'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-07-22T06:43:53.697263+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18598b-49db-6712-8005-bb0c06a0cf7f'}}, tasks=(PregelTask(id='414be979-83cf-c09e-5337-ccabce6936be', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [16]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18598b-49e0-608c-8006-5e9734cc65e0'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-07-22T06:43:53.697263+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18598b-49db-6712-8005-bb0c06a0cf7f'}}, tasks=(PregelTask(id='414be979-83cf-c09e-5337-ccabce6936be', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18598b-49db-6712-8005-bb0c06a0cf7f'}}, metadata={'source': 'loop', 'step': 5, 'parents': {}}, created_at='2026-07-22T06:43:53.695390+00:0

In [18]:
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n Final State : ", final_state)

Step 2 hanging... now manually interrupt from the notebook toolbar (STOP BUTTON)
Step 3 Executed

 Final State :  {'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}


In [19]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18599f-b099-61aa-8008-c886a7ed84c0'}}, metadata={'source': 'loop', 'step': 8, 'parents': {}}, created_at='2026-07-22T06:53:01.339444+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18599f-b094-6c84-8007-3fb49947e4bd'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18599f-b094-6c84-8007-3fb49947e4bd'}}, metadata={'source': 'loop', 'step': 7, 'parents': {}}, created_at='2026-07-22T06:53:01.337658+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f18598b-49e0-608c-8006-5e9734cc65e0'}}, tasks=(PregelTask(id='6